# U-Net Training v2 — Reduced Overfitting

Same model and data pipeline as notebook 16, with four changes aimed at
reducing the validation-loss plateau observed after ~20 epochs:

1. **Stride 64 → 128** — halves the patch overlap, reducing near-duplicate training samples
2. **Target size 512×512 → 256×256** — patches are already 256 pings wide, so this removes the unnecessary 2× upscale
3. **Partial encoder unfreeze** — layers 3–4 of the ResNet34 encoder are trainable (lower LR) so higher-level features can adapt to the sonar domain
4. **Early stopping** — halts training when validation IoU stops improving, with 100 epoch budget

In [ ]:
import sys
sys.path.insert(0, '../../..')

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from deep_learning.training.config import PipelineConfig
from deep_learning.echo_dataset import build_datasets_from_config
from deep_learning.unet_model import (
    create_unet, freeze_encoder_stages, get_trainable_param_count,
    get_param_groups, DiceBCELoss, train_one_epoch, validate,
    predict_full_echogram, EarlyStopping,
)
from deep_learning.evaluate import evaluate_files, plot_evaluation, print_summary

## Configuration

In [ ]:
DATASETS = Path('/Users/axelhagen/Master/Datasets')
ZARR_DIR = DATASETS / 'echopype_zarr'
PATCH_CACHE = DATASETS / 'patch_cache'
CHECKPOINT_DIR = Path('/Users/axelhagen/Master/code/checkpoints')

# Collect all usable zarr stores (those that have cached patches)
zarr_paths = sorted([
    ZARR_DIR / npz.name.split('_')[0].replace('.npz', '.zarr')
    for npz in PATCH_CACHE.glob('*.npz')
])
# Deduplicate and filter to existing
zarr_paths = sorted(set(zp for zp in zarr_paths if zp.exists()))

# If the above heuristic doesn't work (cache filenames include a hash),
# fall back to all zarr stores in the directory
if not zarr_paths:
    zarr_paths = sorted(ZARR_DIR.glob('*.zarr'))

cfg = PipelineConfig(
    zarr_paths=zarr_paths,
    freq1_hz=38_000,
    freq2_hz=200_000,
    patch_width=256,
    stride=128,                    # was 64 — reduces overlap from 75% to 50%
    vmin=-80.0,
    vmax=-34.0,
    target_size=(256, 256),        # was (512, 512) — no upscaling needed
    encoder_name='resnet34',
    freeze_strategy='stages',      # was 'encoder' — partial unfreeze
    freeze_stages=3,               # freeze first 3 stages, unfreeze layer3 + layer4
    in_channels=3,
    batch_size=4,
    num_epochs=100,                # was 30 — early stopping handles when to stop
    encoder_lr=1e-5,               # was 1e-4 — lower LR for unfrozen encoder stages
    decoder_lr=1e-3,
    val_fraction=0.2,
    seed=42,
    checkpoint_dir=CHECKPOINT_DIR,
    cache_dir=PATCH_CACHE,
)

print(f'Device: {cfg.device}')
print(f'Zarr files: {len(zarr_paths)}')
print(f'Checkpoint dir: {cfg.checkpoint_dir}')

## Build datasets

Loads cached patches (stride=128 produces new cache files separate from the stride=64 ones)
and splits at the file level.

In [ ]:
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

train_ds, val_ds, zarr_used = build_datasets_from_config(cfg)

print(f'Training patches:   {len(train_ds)}')
print(f'Validation patches: {len(val_ds)}')
print(f'Zarr files used:    {len(zarr_used)}')

## Inspect a batch

In [ ]:
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

images, masks = next(iter(train_loader))
print(f'Image batch: {images.shape}  dtype={images.dtype}  range=[{images.min():.3f}, {images.max():.3f}]')
print(f'Mask batch:  {masks.shape}  dtype={masks.dtype}  unique={torch.unique(masks).tolist()}')

fig, axes = plt.subplots(2, cfg.batch_size, figsize=(4 * cfg.batch_size, 6))
for i in range(cfg.batch_size):
    axes[0, i].imshow(images[i, 0].numpy(), aspect='auto', cmap='viridis', vmin=0, vmax=1)
    axes[0, i].set_title(f'38 kHz [{i}]')
    axes[1, i].imshow(masks[i].numpy(), aspect='auto', cmap='gray_r', vmin=0, vmax=1)
    axes[1, i].set_title(f'Mask [{i}]')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## Model setup

Partially frozen encoder: stages 0–2 (conv1, layer1, layer2) are fixed, while
layer3 and layer4 are trainable with a lower learning rate.

In [ ]:
device = cfg.device

model = create_unet(encoder_name=cfg.encoder_name, in_channels=cfg.in_channels, classes=1)
freeze_encoder_stages(model, freeze_stages=cfg.freeze_stages)
model = model.to(device)

counts = get_trainable_param_count(model)
print(f'Encoder: {cfg.encoder_name} (first {cfg.freeze_stages} stages frozen)')
print(f'Trainable: {counts["trainable"]:,} / {counts["total"]:,} '
      f'({counts["trainable_pct"]:.1f}%)')
print(f'Input:  ({cfg.in_channels}, {cfg.target_size[0]}, {cfg.target_size[1]})')
print(f'Output: (1, {cfg.target_size[0]}, {cfg.target_size[1]})')

## Training

Early stopping monitors validation IoU (patience=7). Each run gets a timestamped
checkpoint so previous results are never overwritten.

In [ ]:
param_groups = get_param_groups(model, encoder_lr=cfg.encoder_lr, decoder_lr=cfg.decoder_lr)
optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=cfg.lr_factor, patience=cfg.lr_patience)
loss_fn = DiceBCELoss(bce_weight=cfg.bce_weight)
scaler = torch.amp.GradScaler() if cfg.use_amp else None

early_stopping = EarlyStopping(patience=7, min_delta=0.001)

# Timestamped run ID so each training run gets unique files
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = cfg.checkpoint_dir / f'unet_{cfg.encoder_name}_{run_id}_best.pth'
history_path = cfg.checkpoint_dir / f'training_history_{run_id}.json'
metrics_path = cfg.checkpoint_dir / f'eval_metrics_{run_id}.json'

print(f'Run ID: {run_id}')
print(f'Checkpoint: {checkpoint_path}')

history = {
    'train_loss': [], 'val_loss': [],
    'train_iou': [], 'val_iou': [],
    'train_dice': [], 'val_dice': [],
}

best_val_iou = 0.0
best_epoch = -1

for epoch in range(cfg.num_epochs):
    train_m = train_one_epoch(model, train_loader, optimizer, loss_fn, device, scaler)
    val_m = validate(model, val_loader, loss_fn, device)
    scheduler.step(val_m['loss'])

    for key in ['loss', 'iou', 'dice']:
        history[f'train_{key}'].append(train_m[key])
        history[f'val_{key}'].append(val_m[key])

    improved = val_m['iou'] > best_val_iou
    if improved:
        best_val_iou = val_m['iou']
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_iou': best_val_iou,
            'config': {
                'encoder_name': cfg.encoder_name,
                'target_size': cfg.target_size,
                'freq1_hz': cfg.freq1_hz,
                'freq2_hz': cfg.freq2_hz,
                'vmin': cfg.vmin,
                'vmax': cfg.vmax,
                'stride': cfg.stride,
                'freeze_stages': cfg.freeze_stages,
            }
        }, checkpoint_path)

    print(f'Epoch {epoch+1:3d}/{cfg.num_epochs}  '
          f'train_loss={train_m["loss"]:.4f}  '
          f'val_loss={val_m["loss"]:.4f}  '
          f'val_IoU={val_m["iou"]:.4f}  '
          f'val_Dice={val_m["dice"]:.4f}'
          f'{"  *" if improved else ""}')

    if early_stopping(val_m['iou']):
        print(f'\nEarly stopping at epoch {epoch + 1} (no improvement for {early_stopping.patience} epochs)')
        break

print(f'\nBest val IoU: {best_val_iou:.4f} at epoch {best_epoch + 1}')
print(f'Checkpoint: {checkpoint_path}')

## Training curves

In [ ]:
n_epochs_ran = len(history['train_loss'])
epochs = range(1, n_epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
ax1.plot(epochs, history['val_loss'], 'r-', label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Dice + BCE Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_iou'], 'b-', label='Train IoU')
ax2.plot(epochs, history['val_iou'], 'r-', label='Val IoU')
ax2.plot(epochs, history['train_dice'], 'b--', alpha=0.5, label='Train Dice')
ax2.plot(epochs, history['val_dice'], 'r--', alpha=0.5, label='Val Dice')
ax2.axvline(best_epoch + 1, color='gray', linestyle=':', alpha=0.5, label=f'Best (epoch {best_epoch+1})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('IoU and Dice')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'History saved: {history_path}')

## Evaluation

Reload the best checkpoint and run full-echogram inference on validation files.

In [ ]:
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"] + 1} (val IoU={ckpt["val_iou"]:.4f})')

# Determine validation files (same split logic as build_datasets_from_config)
n_files = len(zarr_used)
rng = np.random.RandomState(cfg.seed)
indices = rng.permutation(n_files)
n_val = max(1, int(n_files * cfg.val_fraction))
val_indices = set(indices[:n_val])
val_zarr = [zarr_used[i] for i in range(n_files) if i in val_indices]

print(f'Validation files: {len(val_zarr)}')
for vz in val_zarr:
    print(f'  {vz.name}')

In [ ]:
all_metrics, summary = evaluate_files(
    model, val_zarr,
    freq1_hz=cfg.freq1_hz,
    freq2_hz=cfg.freq2_hz,
    target_size=cfg.target_size,
    device=device,
    patch_width=cfg.patch_width,
    stride=cfg.stride,
    vmin=cfg.vmin,
    vmax=cfg.vmax,
)

print_summary(all_metrics, summary)

with open(metrics_path, 'w') as f:
    json.dump({
        'summary': summary,
        'per_file': [
            {
                'file': m.file_name,
                'iou': m.iou, 'dice': m.dice,
                'precision': m.precision, 'recall': m.recall,
                'bottom_error_mean_m': m.bottom_error_mean_m,
                'bottom_error_median_m': m.bottom_error_median_m,
                'bottom_error_p95_m': m.bottom_error_p95_m,
            }
            for m in all_metrics
        ],
    }, f, indent=2)
print(f'\nMetrics saved: {metrics_path}')

## Visual evaluation

In [ ]:
from deep_learning.evaluate import evaluate_file

n_show = min(3, len(val_zarr))

for vz in val_zarr[:n_show]:
    metrics, prediction, gt_mask, sv_channels, range_arr = evaluate_file(
        model, vz,
        freq1_hz=cfg.freq1_hz,
        freq2_hz=cfg.freq2_hz,
        target_size=cfg.target_size,
        device=device,
        patch_width=cfg.patch_width,
        stride=cfg.stride,
        vmin=cfg.vmin,
        vmax=cfg.vmax,
    )
    plot_evaluation(sv_channels, gt_mask, prediction, range_arr, metrics)
    plt.show()